In [ ]:
!pip install gensim

In [ ]:
import re
import torch
from torch import nn, optim
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
import gensim.downloader as api

In [ ]:
# Get the word embeddings from gensim
glove = api.load('glove-wiki-gigaword-50')

In [ ]:
# Use GPU instead of CPU for faster results
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
dataset = load_dataset('imdb')

In [ ]:
# Function to clean up text and return it word by word (token)
def tokenize(text):
  text = re.sub(r'<.*?>', ' ', text)  # remove HTML tags
  text = text.lower()
  text = text.replace("'", " '").replace("!", " !").replace("?", " ?")
  return text.split()

In [ ]:
class Vocabulary():
  def __init__(self):
    self.wordmap = {'<pad>' : 0,
                    '<unk>' : 1}

  def build(self, sentences):
    for sentence in sentences:
      sentence = tokenize(sentence)
      for word in sentence:
        if word not in self.wordmap.keys():
          self.wordmap[word] = max(self.wordmap.values()) + 1

    return None

  def to_indices(self, text):
    indices = []
    for word in tokenize(text):
      if word not in self.wordmap.keys():
        indices.append(1)
      else:
        indices.append(self.wordmap[word])

    return indices

In [ ]:
# Get the 10k most frequent words for the vocbulary
def extract_common_words(limit=10000):
  reviews = dataset['train']['text']
  word_counts = dict()

  for review in reviews:
    review = tokenize(review)
    for word in review:
      if word not in word_counts.keys():
        word_counts[word] = 1
      else:
        word_counts[word] += 1

  sorted_words = sorted(word_counts, key=word_counts.get, reverse=True)
  most_common_words = sorted_words[:limit]

  return most_common_words

In [ ]:
# Create our vocabulary with the 10k most common words in IMDB reviews
words = extract_common_words()
vocab = Vocabulary()
vocab.build(words)

In [ ]:
# In L18 I used an RNN model. It has the problem of forgetting early words
# due to the vanishing/exploding gradient problem, which makes the weights
# of the first words either not update or update astronomically
# LSTMs fix this issue, by keeping a separate long-term memory
# whose weights are 'immune' to this issue
class LSTMSentimentModel(nn.Module): # Inherit from nn.Module
    # Define the architecture of the model
    def __init__(self):
      super().__init__()
      # Create the embedding table (10000 rows, 1 for each word, each with 50 dimensions)
      self.embedding = nn.Embedding(
                          num_embeddings=len(vocab.wordmap),
                          embedding_dim=50
                      )
      self.lstm = nn.LSTM(input_size=50, hidden_size=128, batch_first=True,
                          dropout=0.3, num_layers=2)
      self.dropout = nn.Dropout(p=0.3)
      self.linear = nn.Linear(128, 1)

    # Define the process of the model
    def forward(self, x):
        x = self.embedding(x) # shape - (n_reviews per batch (64), n_words per review rows(256), embed_dim columns (50))
        out, (hidden, cell) = self.lstm(x) # out is every hidden state, hidden is the very last one
        hidden = hidden[-1]
        hidden = self.dropout(hidden)
        x = torch.sigmoid(self.linear(hidden))

        return x

In [ ]:
def get_review_indices(review):
  indices = vocab.to_indices(review)
  if len(indices) > 256:
    indices = indices[:256]
  else:
    indices += [0] * (256 - len(indices))  # 0 = <pad> index
  return indices

In [ ]:
# Prepare data for training/testing
review_tensors_train = []
labels_train = dataset['train']['label']

review_tensors_test = []
labels_test = dataset['test']['label']

for review in dataset['train']['text']:
  review_tensors_train.append(torch.tensor(get_review_indices(review), dtype=torch.long))

for review in dataset['test']['text']:
  review_tensors_test.append(torch.tensor(get_review_indices(review), dtype=torch.long))

X_train = torch.stack(review_tensors_train)
y_train = torch.tensor(labels_train, dtype=torch.float32).reshape(-1, 1)

X_test = torch.stack(review_tensors_test)
y_test = torch.tensor(labels_test, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# Create the model
model = LSTMSentimentModel()
model = model.to(device)

In [ ]:
# Set the embedding tables to existing words in our table
# To gensim's values for a better optimized start
for word, idx in vocab.wordmap.items():
    if word in glove:
        model.embedding.weight.data[idx] = torch.tensor(glove[word])

In [ ]:
# Define the loss function and the optimizer
loss_ft = nn.BCELoss()
opt = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
n_epochs = 8

for epoch in range(n_epochs):

  batch_loss = 0

  # Train
  model.train()
  for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    pred = model(X_batch)
    loss = loss_ft(pred, y_batch)
    loss.backward()
    opt.step()
    opt.zero_grad()

    batch_loss += loss.item()

  avg_batch_loss = batch_loss / len(train_loader)


  print(f'Epoch {epoch}')
  print(f'Loss: {avg_batch_loss:.4f}')

Epoch 0
Loss: 0.6935
Epoch 1
Loss: 0.5313
Epoch 2
Loss: 0.3752
Epoch 3
Loss: 0.3374
Epoch 4
Loss: 0.2825
Epoch 5
Loss: 0.2557
Epoch 6
Loss: 0.2315
Epoch 7
Loss: 0.2105


In [ ]:
model.eval()
total_correct = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds = (model(X_batch) > 0.5).float()
        total_correct += (preds == y_batch).sum().item()
print(f"Test accuracy: {total_correct / len(test_dataset):.4f}")

Test accuracy: 0.8334
